<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_5_Hybrid_NormalizingFlow_DensityRatio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 5 — Hybrid normalizing-flow and density-ratio estimation

Exercise 4 estimated the signal and background densities with two independent normalizing flows. That is flexible, but the likelihood ultimately depends on the much more demanding ratio $p_S(x)/p_B(x)$, so small unrelated flow errors can be amplified.

Here we combine the two approaches introduced earlier in the workshop:

1. use the same learned PRESEL selection as Exercise 4 to improve $S/B$;
2. build an equal-mixture reference density after PRESEL,
   $$p_{\rm ref}(x)=\tfrac12 p_S(x)+\tfrac12 p_B(x);$$
3. train **one** normalizing flow for $p_{\rm ref}$;
4. sample one million unweighted reference events from that flow;
5. use the Exercise 2 classifier method to estimate
   $$r_S(x)=\frac{p_S(x)}{p_{\rm ref}(x)},\qquad
     r_B(x)=\frac{p_B(x)}{p_{\rm ref}(x)};$$
6. reconstruct the component densities as
   $$\hat p_S(x)=\hat p_{\rm ref}(x)\hat r_S(x),\qquad
     \hat p_B(x)=\hat p_{\rm ref}(x)\hat r_B(x);$$
7. repeat the extended-likelihood fit and toy study from Exercise 4.

The reference flow provides flexible generation and an explicit density. The classifiers only need to learn ratios against a reference that covers signal and background equally. In the profile likelihood, common errors in $p_{\rm ref}$ are independent of $\mu$ and cancel, while the precision-critical information is learned discriminatively.


In [ ]:
## ============================================================================
# Google Colab setup — run me first.  Safe to re-run; a no-op off Colab.
# ============================================================================
import os, sys

# --- config -----------------------------------------------------------------
REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"  # package + tutorial helpers
BRANCH   = "ml4hep_school_tutorial"
N_BKG, N_SIG = 100_000_000, 20_000_000     # Colab-sized dataset (raise for less MC noise in the fit)
USE_DRIVE = True                    # True -> save data/models to Google Drive so they
                                    # persist across notebooks & sessions (see notes above)
REMAKE_EVENTS = False
# ----------------------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab"
    else:
        ROOT = "/content"
    os.makedirs(ROOT, exist_ok=True)
    os.chdir(ROOT)

    # 1) fetch ONLY the package source + tutorial helpers (skip Git-LFS / big blobs)
    if not os.path.isdir("nsbi-lhc-toolkit"):
        os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
        !git clone --depth 1 --filter=blob:none --sparse --branch $BRANCH $REPO_URL
        !cd nsbi-lhc-toolkit && git sparse-checkout set src workshops/ml4hep_tifr
    else:
        !git -C nsbi-lhc-toolkit fetch origin $BRANCH
        !git -C nsbi-lhc-toolkit checkout $BRANCH
        !git -C nsbi-lhc-toolkit pull --ff-only origin $BRANCH

    # 2) make `import nsbi_common_utils` work (pure-python src layout, no build step)
    src = os.path.abspath("nsbi-lhc-toolkit/src")
    if src not in sys.path:
        sys.path.insert(0, src)

    # 3) runtime deps Colab doesn't already ship (torch/jax/sklearn/... are preinstalled)
    !pip install -q pytorch-lightning onnx onnxruntime onnxscript iminuit mplhep nflows pyarrow

    # 4) work from the tutorial dir so utils.py / generate_distributions.py and the
    #    ./dataframes, ./models_* relative paths resolve just like a local run
    os.chdir("nsbi-lhc-toolkit/workshops/ml4hep_tifr")

    # 5) generate the Gaussian-mixture samples if they aren't there yet
    if REMAKE_EVENTS or (not os.path.exists("dataframes/signal.parquet")):
        !python generate_distributions.py --n_bkg $N_BKG --n_sig $N_SIG

print("Working dir:", os.getcwd())


## Inputs expected by this notebook

The notebook uses the same generated files as Exercise 4:

```text
dataframes/background.parquet
dataframes/signal.parquet
```

Only the reconstructed variables `x1,...,x5` enter PRESEL, the reference flow, the density-ratio networks, and the final likelihood. The parquet files are streamed so the complete event samples are never loaded into memory.


In [ ]:
import gc
import math
import os
from contextlib import nullcontext, redirect_stdout
from io import StringIO
from pathlib import Path

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import multivariate_normal

import torch

import nsbi_common_utils
from nsbi_common_utils.training import density_ratio_trainer, predict_with_model

from utils import (
    FEATURES,
    background_components,
    signal_components,
    smearing_parameters,
)
from utils_nf import (
    accumulate_preselection_histogram,
    choose_preselection_ratio_cut,
    collect_preselected_parquet,
    flow_log_prob_x,
    flow_sample_x,
    sample_parquet_partition,
    train_flow,
)
from utils_plotting import (
    plot_flow_pair_closure,
    plot_log_density_truth_binned,
    plot_log_density_truth_scatter,
    plot_log_prob_cdf_closure,
    plot_log_prob_closure,
    plot_mu_hat_toys,
    plot_profile_scan,
    plot_profile_scan_comparison,
    plot_t_mu_toys,
)

FEATURES = list(FEATURES)
N_DIM = len(FEATURES)

SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Features: {FEATURES}")


In [ ]:
BASE_PATH = Path("./dataframes")
PRESEL_MODEL_DIR = Path("models_PRESEL")
PRESEL_PLOT_DIR = Path("plots_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference")
REFERENCE_FLOW_PLOT_DIR = Path("plots_flows_hybrid_reference")
HYBRID_DENSITY_DIR = Path("saved_densities_hybrid")
HYBRID_PLOT_DIR = Path("plots_hybrid")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef"),
    "background": Path("models_Hybrid_BkgvsRef"),
}
RATIO_PLOT_DIR = {
    "signal": Path("plots_Hybrid_SigvsRef"),
    "background": Path("plots_Hybrid_BkgvsRef"),
}

for directory in [
    PRESEL_MODEL_DIR,
    PRESEL_PLOT_DIR,
    REFERENCE_FLOW_MODEL_DIR,
    REFERENCE_FLOW_PLOT_DIR,
    HYBRID_DENSITY_DIR,
    HYBRID_PLOT_DIR,
    *RATIO_MODEL_DIR.values(),
    *RATIO_PLOT_DIR.values(),
]:
    directory.mkdir(parents=True, exist_ok=True)

# The three partitions are identical to Exercise 4.
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.5
STREAM_BATCH_SIZE = 100_000
PRESEL_CUT_HISTOGRAM_BINS = 4_000
PRESEL_LOG_RATIO_RANGE = (-20.0, 20.0)

# PRESEL classifier and target.
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 250.0
PRESEL_MAX_TRAIN_EVENTS_PER_CLASS = 500_000
PRESEL_HIDDEN_LAYERS = 3
PRESEL_NEURONS = 256
PRESEL_N_EPOCHS = 30
PRESEL_BATCH_SIZE = 2048
PRESEL_LEARNING_RATE = 1.0e-3
PRESEL_HOLDOUT_FRACTION = 0.25
PRESEL_VALIDATION_FRACTION = 0.20
PRESEL_PATIENCE = 8
PRESEL_LOAD_IF_AVAILABLE = True

# Bounded post-selection reservoirs.
MAX_TRAIN_EVENTS = {"background": 1_000_000, "signal": 1_000_000}
MAX_EVAL_EVENTS = {"background": 250_000, "signal": 250_000}

# Equal event counts implement the 50:50 reference normalization for flow
# training, because train_flow performs unweighted maximum likelihood.
REFERENCE_COMPONENT_TRAIN_EVENTS = 500_000
REFERENCE_COMPONENT_EVAL_EVENTS = 125_000
N_REFERENCE_EVENTS = 1_000_000
REFERENCE_SAMPLING_BATCH_SIZE = 65_536

# The reference flow uses exactly the same implementation and defaults as the
# spline flow selected in Exercise 4.
USE_QUADRATIC_SPLINE = True
FLOW_TYPE = "quadratic_spline" if USE_QUADRATIC_SPLINE else "realnvp"
N_COUPLING_LAYERS = 10
HIDDEN_FEATURES = 1024
HIDDEN_LAYERS = 4
SCALE_CLIP = 1.5
SPLINE_NUM_BINS = 8
SPLINE_TAIL_BOUND = 3.0
DROPOUT_PROBABILITY = 0.0

BATCH_SIZE = 2048
N_EPOCHS = 70
LEARNING_RATE = 1.0e-4
LR_SCHEDULER_FACTOR = 0.2
LR_SCHEDULER_PATIENCE = 2
MIN_LEARNING_RATE = 1.0e-7
WEIGHT_DECAY = 0.0
VALIDATION_FRACTION = 0.20
PATIENCE = 5
GRADIENT_CLIP = 5.0
FLOW_LOAD_IF_AVAILABLE = True

MODEL_CONFIG = {
    "flow_type": FLOW_TYPE,
    "n_features": N_DIM,
    "n_coupling_layers": N_COUPLING_LAYERS,
    "hidden_features": HIDDEN_FEATURES,
    "hidden_layers": HIDDEN_LAYERS,
    "scale_clip": SCALE_CLIP,
    "spline_num_bins": SPLINE_NUM_BINS,
    "spline_tail_bound": SPLINE_TAIL_BOUND,
    "dropout_probability": DROPOUT_PROBABILITY,
}
TRAINING_CONFIG = {
    "batch_size": BATCH_SIZE,
    "n_epochs": N_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "lr_scheduler_factor": LR_SCHEDULER_FACTOR,
    "lr_scheduler_patience": LR_SCHEDULER_PATIENCE,
    "min_learning_rate": MIN_LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "validation_fraction": VALIDATION_FRACTION,
    "patience": PATIENCE,
    "gradient_clip": GRADIENT_CLIP,
}

# Exercise 2-style classifier settings for both ratios.
MAX_RATIO_EVENTS_PER_CLASS = 750_000
RATIO_HIDDEN_LAYERS = 4
RATIO_NEURONS = 1024
RATIO_N_EPOCHS = 50
RATIO_BATCH_SIZE = 4096
RATIO_LEARNING_RATE = 1.0e-3
RATIO_HOLDOUT_FRACTION = 0.25
RATIO_VALIDATION_FRACTION = 0.20
RATIO_PATIENCE = 10
RATIO_LOAD_IF_AVAILABLE = True
RATIO_EVALUATION_BATCH_SIZE = 100_000
RATIO_FLOOR = 1.0e-12

print(f"Selected reference-flow architecture: {FLOW_TYPE}")


## Stream data into independent, bounded samples

As in Exercise 4, a deterministic row hash assigns events to PRESEL training, post-selection model training, or final evaluation. Only capped random reservoirs are kept, while yields and efficiencies are accumulated over all streamed events.


In [ ]:
SAMPLE_PATHS = {
    "signal": BASE_PATH / "signal.parquet",
    "background": BASE_PATH / "background.parquet",
}

# First pass: retain only the bounded samples needed for PRESEL training.
PRESEL_signal_bce, PRESEL_signal_input_stats = sample_parquet_partition(
    SAMPLE_PATHS["signal"],
    features=FEATURES,
    partition="presel",
    max_events=PRESEL_MAX_TRAIN_EVENTS_PER_CLASS,
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
    reservoir_seed=SEED + 11,
)
PRESEL_background_bce, PRESEL_background_input_stats = sample_parquet_partition(
    SAMPLE_PATHS["background"],
    features=FEATURES,
    partition="presel",
    max_events=PRESEL_MAX_TRAIN_EVENTS_PER_CLASS,
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
    reservoir_seed=SEED + 22,
)


In [ ]:
PRESEL_INPUT_STATS = {
    "signal": PRESEL_signal_input_stats,
    "background": PRESEL_background_input_stats,
}
PRESEL_INCLUSIVE_YIELD = {
    sample_name: float(stats["inclusive_weight"])
    for sample_name, stats in PRESEL_INPUT_STATS.items()
}

for sample_name, stats in PRESEL_INPUT_STATS.items():
    print(
        f"{sample_name:10s}: {stats['inclusive_events']:,} total events, "
        f"sum weights = {stats['inclusive_weight']:.6g}; "
        f"retained {stats['retained_events']:,} for PRESEL"
    )
print(
    "Inclusive B/S = "
    f"{PRESEL_INCLUSIVE_YIELD['background'] / PRESEL_INCLUSIVE_YIELD['signal']:,.1f}"
)


## Preselection with a learned density ratio

The PRESEL classifier estimates

$$r_{\rm PRESEL}(x)=\frac{p_S(x)}{p_B(x)}.$$

Its training and its target post-selection $B/S$ are intentionally identical to Exercise 4. PRESEL is not part of the final hybrid likelihood; it defines the phase-space region in which all subsequent densities and ratios live.


In [ ]:
for PRESEL_sample, PRESEL_label in [
    (PRESEL_signal_bce, 1.0),
    (PRESEL_background_bce, 0.0),
]:
    PRESEL_sample["PRESEL_label"] = PRESEL_label
    PRESEL_sample["PRESEL_weight"] = (
        PRESEL_sample["weight"] / PRESEL_sample["weight"].sum()
    )

PRESEL_training_dataframe = pd.concat(
    [PRESEL_signal_bce, PRESEL_background_bce],
    ignore_index=True,
).sample(frac=1.0, random_state=SEED, ignore_index=True)

PRESEL_trainer = density_ratio_trainer(
    dataset=PRESEL_training_dataframe,
    weights=PRESEL_training_dataframe["PRESEL_weight"].to_numpy(),
    training_labels=PRESEL_training_dataframe["PRESEL_label"].to_numpy(),
    features=FEATURES,
    features_scaling=FEATURES,
    sample_name=["signal", "background"],
    output_name="PRESEL",
    path_to_figures=f"{PRESEL_PLOT_DIR}/",
    path_to_models=f"{PRESEL_MODEL_DIR}/",
)

PRESEL_trainer.train(
    hidden_layers=PRESEL_HIDDEN_LAYERS,
    neurons=PRESEL_NEURONS,
    number_of_epochs=PRESEL_N_EPOCHS,
    batch_size=PRESEL_BATCH_SIZE,
    learning_rate=PRESEL_LEARNING_RATE,
    scalerType="MinMax",
    ensemble_index=0,
    verbose=1,
    rnd_seed=SEED,
    holdout_split=PRESEL_HOLDOUT_FRACTION,
    validation_split=PRESEL_VALIDATION_FRACTION,
    callback_patience=PRESEL_PATIENCE,
    num_workers=0,
    load_trained_models=PRESEL_LOAD_IF_AVAILABLE,
    calibration=False,
)

# Keep one reusable ONNX Runtime session for every streamed prediction batch.
PRESEL_scaler = PRESEL_trainer.scaler
PRESEL_model_candidate = PRESEL_trainer.model_NN
if isinstance(PRESEL_model_candidate, ort.InferenceSession):
    PRESEL_model = PRESEL_model_candidate
else:
    PRESEL_available_providers = ort.get_available_providers()
    PRESEL_providers = [
        provider
        for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if provider in PRESEL_available_providers
    ]
    if not PRESEL_providers:
        PRESEL_providers = PRESEL_available_providers
    PRESEL_session_options = ort.SessionOptions()
    PRESEL_session_options.intra_op_num_threads = 1
    PRESEL_session_options.inter_op_num_threads = 1
    PRESEL_model = ort.InferenceSession(
        PRESEL_model_candidate.SerializeToString(),
        sess_options=PRESEL_session_options,
        providers=PRESEL_providers,
    )

# The trainer holds its dataset, holdout copies, and diagnostic predictions.
# None are needed once the scaler and ONNX session have been extracted.
del (
    PRESEL_trainer,
    PRESEL_model_candidate,
    PRESEL_training_dataframe,
    PRESEL_signal_bce,
    PRESEL_background_bce,
    PRESEL_sample,
)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Choose the PRESEL cut and collect the post-selection reservoirs

The cut is selected from a streamed weighted histogram, after which capped signal and background samples are retained for model training and independent likelihood evaluation. Unlike Exercise 4, the PRESEL inference session remains available so generated reference events can be required to pass the same cut.


In [ ]:
def evaluate_PRESEL_ratio(feature_dataframe):
    ratio = predict_with_model(
        feature_dataframe.astype("float32", copy=False),
        scaler=PRESEL_scaler,
        model=PRESEL_model,
    )
    return np.asarray(ratio, dtype=float).reshape(-1)


PRESEL_LOG_RATIO_EDGES = np.linspace(
    PRESEL_LOG_RATIO_RANGE[0],
    PRESEL_LOG_RATIO_RANGE[1],
    PRESEL_CUT_HISTOGRAM_BINS + 1,
)

# Second pass: determine the cut from small accumulated histograms.
PRESEL_signal_histogram, PRESEL_signal_hist_stats = accumulate_preselection_histogram(
    SAMPLE_PATHS["signal"],
    features=FEATURES,
    ratio_predictor=evaluate_PRESEL_ratio,
    log_ratio_edges=PRESEL_LOG_RATIO_EDGES,
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
)
PRESEL_background_histogram, PRESEL_background_hist_stats = accumulate_preselection_histogram(
    SAMPLE_PATHS["background"],
    features=FEATURES,
    ratio_predictor=evaluate_PRESEL_ratio,
    log_ratio_edges=PRESEL_LOG_RATIO_EDGES,
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
)

PRESEL_RATIO_CUT, PRESEL_CUT_DIAGNOSTICS = choose_preselection_ratio_cut(
    PRESEL_signal_histogram,
    PRESEL_background_histogram,
    PRESEL_LOG_RATIO_EDGES,
    signal_inclusive_yield=PRESEL_INCLUSIVE_YIELD["signal"],
    background_inclusive_yield=PRESEL_INCLUSIVE_YIELD["background"],
    signal_partition_weight=PRESEL_signal_hist_stats["partition_weight"],
    background_partition_weight=PRESEL_background_hist_stats["partition_weight"],
    target_background_to_signal=PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
)

# Third pass: keep only selected, capped train/evaluation reservoirs.
PRESEL_signal_samples, PRESEL_signal_stream_stats = collect_preselected_parquet(
    SAMPLE_PATHS["signal"],
    features=FEATURES,
    ratio_predictor=evaluate_PRESEL_ratio,
    ratio_cut=PRESEL_RATIO_CUT,
    max_train_events=MAX_TRAIN_EVENTS["signal"],
    max_eval_events=MAX_EVAL_EVENTS["signal"],
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
    reservoir_seed=SEED + 101,
)
PRESEL_background_samples, PRESEL_background_stream_stats = collect_preselected_parquet(
    SAMPLE_PATHS["background"],
    features=FEATURES,
    ratio_predictor=evaluate_PRESEL_ratio,
    ratio_cut=PRESEL_RATIO_CUT,
    max_train_events=MAX_TRAIN_EVENTS["background"],
    max_eval_events=MAX_EVAL_EVENTS["background"],
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
    reservoir_seed=SEED + 202,
)

PRESEL_STREAM_STATS = {
    "signal": PRESEL_signal_stream_stats,
    "background": PRESEL_background_stream_stats,
}
PRESEL_SELECTED_YIELD = {}
PRESEL_EFFICIENCY = {}
PRESEL_EVAL_YIELD_BEFORE_RESCALE = {}
for sample_name, stats in PRESEL_STREAM_STATS.items():
    train_stats = stats["flow_train"]
    eval_stats = stats["eval"]
    if train_stats["partition_weight"] <= 0.0 or eval_stats["partition_weight"] <= 0.0:
        raise RuntimeError(f"Empty streamed partition for {sample_name}.")

    PRESEL_EFFICIENCY[sample_name] = (
        train_stats["selected_weight"] / train_stats["partition_weight"]
    )
    PRESEL_SELECTED_YIELD[sample_name] = (
        PRESEL_INCLUSIVE_YIELD[sample_name] * PRESEL_EFFICIENCY[sample_name]
    )
    PRESEL_EVAL_YIELD_BEFORE_RESCALE[sample_name] = (
        PRESEL_INCLUSIVE_YIELD[sample_name]
        * eval_stats["selected_weight"]
        / eval_stats["partition_weight"]
    )

signal_train = PRESEL_signal_samples["flow_train"]
signal_eval = PRESEL_signal_samples["eval"]
background_train = PRESEL_background_samples["flow_train"]
background_eval = PRESEL_background_samples["eval"]

# Each retained reservoir represents the complete selected physical yield.
for sample_name, train_sample, eval_sample in [
    ("signal", signal_train, signal_eval),
    ("background", background_train, background_eval),
]:
    for split_name, sample in [("train", train_sample), ("eval", eval_sample)]:
        retained_weight = float(sample["weight"].sum())
        if len(sample) == 0 or retained_weight <= 0.0:
            raise RuntimeError(
                f"PRESEL retained no {sample_name} events for {split_name}."
            )
        sample["weight"] *= PRESEL_SELECTED_YIELD[sample_name] / retained_weight

TOTAL_YIELD = PRESEL_SELECTED_YIELD.copy()
PRESEL_train_background_to_signal = (
    PRESEL_SELECTED_YIELD["background"] / PRESEL_SELECTED_YIELD["signal"]
)
PRESEL_eval_background_to_signal = (
    PRESEL_EVAL_YIELD_BEFORE_RESCALE["background"]
    / PRESEL_EVAL_YIELD_BEFORE_RESCALE["signal"]
)

print(f"PRESEL ratio cut: r >= {PRESEL_RATIO_CUT:.5g}")
print(
    "Histogram estimate at the cut: B/S = "
    f"{PRESEL_CUT_DIAGNOSTICS['histogram_background_to_signal']:.2f}"
)
print(
    "Signal efficiency = "
    f"{PRESEL_EFFICIENCY['signal']:.3%}; background efficiency = "
    f"{PRESEL_EFFICIENCY['background']:.3%}"
)
print(
    f"Exact post-selection B/S: flow training = {PRESEL_train_background_to_signal:.2f}, "
    f"independent evaluation = {PRESEL_eval_background_to_signal:.2f}"
)
for sample_name, stats in PRESEL_STREAM_STATS.items():
    print(
        f"{sample_name:10s}: retained "
        f"{stats['flow_train']['retained_events']:,}/"
        f"{stats['flow_train']['selected_events']:,} selected train events and "
        f"{stats['eval']['retained_events']:,}/"
        f"{stats['eval']['selected_events']:,} selected eval events"
    )
print("Post-selection yields:", TOTAL_YIELD)

del (
    PRESEL_signal_samples,
    PRESEL_background_samples,
    PRESEL_signal_histogram,
    PRESEL_background_histogram,
)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Construct the equal-mixture reference sample for flow training

After PRESEL, the physical yields still differ by approximately a factor of 250. Training a flow with those physical weights would therefore make the reference almost identical to the background.

Instead, we define

$$
p_{\rm ref}(x)=\frac12 p_S(x\mid\mathrm{pass})+\frac12 p_B(x\mid\mathrm{pass}).
$$

The flow trainer performs ordinary unweighted maximum likelihood, so we realize this definition with equal numbers of signal and background events. The `weight` column is also set so each component integrates to $1/2$ and the complete reference sample integrates to one.


In [ ]:
def make_balanced_reference(signal_df, background_df, n_per_component, seed):
    n_per_component = min(
        int(n_per_component), len(signal_df), len(background_df)
    )
    if n_per_component < 1:
        raise ValueError("Both reference components must contain events.")

    signal_part = signal_df.sample(
        n=n_per_component, random_state=seed
    )[FEATURES].copy()
    background_part = background_df.sample(
        n=n_per_component, random_state=seed + 1
    )[FEATURES].copy()
    signal_part["reference_component"] = "signal"
    background_part["reference_component"] = "background"
    signal_part["weight"] = 0.5 / n_per_component
    background_part["weight"] = 0.5 / n_per_component

    return pd.concat(
        [signal_part, background_part], ignore_index=True
    ).sample(frac=1.0, random_state=seed + 2, ignore_index=True)


reference_flow_train = make_balanced_reference(
    signal_train,
    background_train,
    REFERENCE_COMPONENT_TRAIN_EVENTS,
    SEED + 301,
)
reference_flow_eval = make_balanced_reference(
    signal_eval,
    background_eval,
    REFERENCE_COMPONENT_EVAL_EVENTS,
    SEED + 401,
)

print(
    f"Reference-flow training sample: {len(reference_flow_train):,} events; "
    f"sum weights = {reference_flow_train['weight'].sum():.6f}"
)
print(reference_flow_train.groupby("reference_component")["weight"].sum())


## Train the reference normalizing flow

This is the same `train_flow` interface and the same selectable RealNVP/spline implementation used in Exercise 4. The only change is the target distribution: one flow now learns the balanced reference instead of training separate absolute-density models for signal and background.


In [ ]:
reference_flow = train_flow(
    "reference",
    reference_flow_train,
    features=FEATURES,
    model_dir=REFERENCE_FLOW_MODEL_DIR,
    model_config=MODEL_CONFIG,
    training_config=TRAINING_CONFIG,
    device=device,
    max_train_events=None,
    load_if_available=FLOW_LOAD_IF_AVAILABLE,
    seed=SEED + 501,
)

del reference_flow_train
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Draw the fixed one-million-event reference sample

A smooth flow can leak a small amount of probability outside a hard PRESEL boundary. We therefore sample the flow and apply the same PRESEL classifier and cut by rejection. If $A_{\rm ref}$ is the accepted fraction, the density represented by the retained events is

$$
\hat p_{\rm ref}(x\mid\mathrm{pass})
=\frac{\hat q_{\rm flow}(x)}{A_{\rm ref}},\qquad x\in\mathrm{PRESEL}.
$$

The retained events all receive the same weight $1/N_{\rm ref}$, giving an unweighted reference sample with total normalization one. This single sample is kept for ratio training, normalization checks, weighted signal/background sampling, and pseudo-experiments.


In [ ]:
def sample_preselected_reference_flow(flow_pack, n_events, batch_size=65_536):
    accepted_chunks = []
    n_kept = 0
    n_generated = 0
    n_passed = 0

    while n_kept < int(n_events):
        current_batch = max(
            int(batch_size),
            min(int(n_events) - n_kept, 4 * int(batch_size)),
        )
        generated = flow_sample_x(flow_pack, current_batch, batch_size=batch_size)
        generated_df = pd.DataFrame(generated, columns=FEATURES)
        ratio = evaluate_PRESEL_ratio(generated_df)
        passes = ratio >= PRESEL_RATIO_CUT

        n_generated += len(generated)
        n_passed += int(passes.sum())
        if np.any(passes):
            accepted = generated[passes]
            accepted_chunks.append(accepted)
            n_kept += len(accepted)

    accepted = np.concatenate(accepted_chunks, axis=0)[: int(n_events)]
    acceptance = n_passed / n_generated
    if not 0.0 < acceptance <= 1.0:
        raise RuntimeError("Invalid reference-flow PRESEL acceptance.")
    return accepted.astype(np.float32, copy=False), float(acceptance)


reference_values, REFERENCE_FLOW_PRESEL_ACCEPTANCE = sample_preselected_reference_flow(
    reference_flow,
    N_REFERENCE_EVENTS,
    batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
)
reference_sample = pd.DataFrame(reference_values, columns=FEATURES)
reference_sample["weight"] = 1.0 / len(reference_sample)
del reference_values

print(f"Reference events retained: {len(reference_sample):,}")
print(f"Reference-flow PRESEL acceptance: {REFERENCE_FLOW_PRESEL_ACCEPTANCE:.4%}")
print(f"Reference-sample normalization: {reference_sample['weight'].sum():.6f}")


def reference_log_prob_x(x, batch_size=65_536):
    return (
        np.asarray(
            flow_log_prob_x(reference_flow, x, batch_size=batch_size),
            dtype=np.float64,
        )
        - np.log(REFERENCE_FLOW_PRESEL_ACCEPTANCE)
    )


## Validation 1: feature and correlation closure

The reference flow must reproduce not only every one-dimensional feature distribution, but also the dependence among features. The pair plot below summarizes both:

- **diagonal:** overlaid one-dimensional densities for held-out MC and flow-generated events;
- **lower triangle:** shared-range two-dimensional density contours enclosing approximately 68% and 95% of each sample;
- **upper triangle:** the Pearson correlations $\rho_\mathrm{MC}$ and $\rho_\mathrm{flow}$, together with $\Delta\rho = \rho_\mathrm{flow}-\rho_\mathrm{MC}$.

The contours are particularly important: matching all diagonal projections does not guarantee that the flow has learned the joint five-dimensional structure.


In [ ]:
n_plot = min(50_000, len(reference_flow_eval), len(reference_sample))
reference_pair_mc = reference_flow_eval.sample(
    n=n_plot, random_state=SEED
)[FEATURES].to_numpy(dtype=np.float32)
reference_pair_generated = reference_sample.sample(
    n=n_plot, random_state=SEED + 1
)[FEATURES].to_numpy(dtype=np.float32)

fig = plot_flow_pair_closure(
    "balanced reference",
    reference_pair_mc,
    reference_pair_generated,
    FEATURES,
)
plt.show()


## Validation 2: closure as a function of the learned log density

The five reconstructed variables can also be compressed to the scalar learned log density

$$
\ell(x) = \log \hat p_\mathrm{flow}(x).
$$

Below, we evaluate the **trained reference flow** on two sets of events: held-out MC events and events sampled from the flow. If the learned density reproduces the data distribution, the two distributions of $\ell(x)$ should agree.

The generated reference uses many more events by default than the held-out MC points. Its finite-sampling fluctuations are therefore small, and the visible statistical uncertainty is dominated by the finite MC evaluation sample. The lower panel shows the bin-by-bin ratio $\mathrm{MC}/\mathrm{flow}$; its error bars include the finite counts in both histograms, although the high-statistics flow contribution is small. This does **not** remove uncertainty or bias in the trained flow itself; it only makes the numerical sampling of that fixed flow very precise.

### CDF-versus-CDF closure

The same comparison can be displayed as a probability–probability (P–P) plot. For common thresholds in $\ell=\log \hat p(x)$, we plot the empirical MC CDF against the flow CDF,

$$
F_\mathrm{MC}(\ell) \quad\text{versus}\quad F_\mathrm{flow}(\ell).
$$

Perfect closure therefore follows the diagonal $F_\mathrm{MC}=F_\mathrm{flow}$. The lower panel shows the ratio $F_\mathrm{MC}/F_\mathrm{flow}$; the scan starts at $F_\mathrm{flow}=10^{-3}$ to avoid division by zero. The blue bands are the approximate pointwise $\pm1\sigma$ finite-sample expectations; neighboring CDF points are correlated, and the bands do not include flow-training uncertainty.

This remains a one-dimensional projection. It is a useful and sensitive diagnostic, but agreement here alone does not prove complete five-dimensional closure because different regions of feature space can have the same value of $\log \hat p(x)$.


In [ ]:
n_mc = min(50_000, len(reference_flow_eval))
reference_log_p_mc = reference_log_prob_x(
    reference_flow_eval.sample(n=n_mc, random_state=SEED)[FEATURES]
)
reference_log_p_generated = reference_log_prob_x(reference_sample[FEATURES])

fig = plot_log_prob_closure(
    "balanced reference", reference_log_p_mc, reference_log_p_generated
)
plt.show()
cdf_fig = plot_log_prob_cdf_closure(
    "balanced reference",
    reference_log_p_mc,
    reference_log_p_generated,
    color="C2",
)
plt.show()


## Validation 3: compare the reference flow with analytic truth

The smeared signal and background densities are analytically available for this toy generator. After PRESEL, the truth reference is

$$
p_{\rm ref}^{\rm truth}(x)
=\tfrac12\frac{p_S(x)}{\epsilon_S}
+\tfrac12\frac{p_B(x)}{\epsilon_B},\qquad x\in\mathrm{PRESEL}.
$$

This comparison diagnoses the reference flow itself. Later we will verify that ratio multiplication cancels residual reference-flow errors in the reconstructed signal and background densities.


In [ ]:
def reco_components_from_truth_components(components):
    scale, resolution = smearing_parameters()
    scale = np.asarray(scale, dtype=float)
    resolution = np.asarray(resolution, dtype=float)
    D = np.diag(scale)
    response_cov = np.diag(resolution**2)

    reco_components = []
    for frac, mean_y, cov_y in components:
        mean_x = scale * np.asarray(mean_y, dtype=float)
        cov_x = D @ np.asarray(cov_y, dtype=float) @ D.T + response_cov
        reco_components.append((frac, mean_x, cov_x))
    return reco_components


def mixture_log_density(x, components):
    x = np.asarray(x, dtype=float)
    fracs = np.asarray([component[0] for component in components], dtype=float)
    fracs = fracs / fracs.sum()
    terms = [
        np.log(frac)
        + multivariate_normal(mean=mean, cov=cov, allow_singular=False).logpdf(x)
        for frac, (_, mean, cov) in zip(fracs, components)
    ]
    return logsumexp(np.vstack(terms), axis=0)


TRUTH_RECO_COMPONENTS = {
    "background": reco_components_from_truth_components(background_components()),
    "signal": reco_components_from_truth_components(signal_components()),
}


def selected_truth_log_density(x, sample_name):
    return (
        mixture_log_density(x, TRUTH_RECO_COMPONENTS[sample_name])
        - np.log(PRESEL_EFFICIENCY[sample_name])
    )


def reference_truth_log_density(x):
    return np.logaddexp(
        np.log(0.5) + selected_truth_log_density(x, "signal"),
        np.log(0.5) + selected_truth_log_density(x, "background"),
    )


def build_binned_log_density_calibration(log_p_truth, log_p_model, n_bins=25):
    log_p_truth = np.asarray(log_p_truth, dtype=float)
    log_p_model = np.asarray(log_p_model, dtype=float)
    delta = log_p_model - log_p_truth
    edges = np.unique(np.quantile(log_p_truth, np.linspace(0.0, 1.0, n_bins + 1)))
    eps = 1.0e-9 * max(1.0, float(edges[-1] - edges[0]))
    edges = edges.copy()
    edges[0] -= eps
    edges[-1] += eps
    index = np.clip(
        np.searchsorted(edges, log_p_truth, side="right") - 1,
        0,
        len(edges) - 2,
    )

    rows = []
    for bin_index in range(len(edges) - 1):
        mask = index == bin_index
        count = int(mask.sum())
        if count == 0:
            rows.append(
                {
                    "bin": bin_index,
                    "truth_lo": edges[bin_index],
                    "truth_hi": edges[bin_index + 1],
                    "count": 0,
                    "truth_mean": np.nan,
                    "flow_mean": np.nan,
                    "flow_sem": np.nan,
                    "delta_mean": np.nan,
                    "delta_sem": np.nan,
                }
            )
            continue
        flow_std = log_p_model[mask].std(ddof=1) if count > 1 else 0.0
        delta_std = delta[mask].std(ddof=1) if count > 1 else 0.0
        rows.append(
            {
                "bin": bin_index,
                "truth_lo": edges[bin_index],
                "truth_hi": edges[bin_index + 1],
                "count": count,
                "truth_mean": log_p_truth[mask].mean(),
                "flow_mean": log_p_model[mask].mean(),
                "flow_sem": flow_std / np.sqrt(count),
                "delta_mean": delta[mask].mean(),
                "delta_sem": delta_std / np.sqrt(count),
            }
        )
    return pd.DataFrame(rows), edges


In [ ]:
n_truth_validation = min(100_000, len(reference_flow_eval))
X_reference_validation = reference_flow_eval.sample(
    n=n_truth_validation, random_state=SEED
)[FEATURES].to_numpy(dtype=np.float32)
reference_log_p_truth = reference_truth_log_density(X_reference_validation)
reference_log_p_flow = reference_log_prob_x(X_reference_validation)
reference_delta = reference_log_p_flow - reference_log_p_truth

print(
    "reference: "
    f"corr(log p) = {np.corrcoef(reference_log_p_truth, reference_log_p_flow)[0, 1]:.4f}, "
    f"RMSE = {np.sqrt(np.mean(reference_delta**2)):.4f}, "
    f"bias = {reference_delta.mean():.4f}"
)
fig = plot_log_density_truth_scatter(
    "balanced reference", reference_log_p_truth, reference_log_p_flow
)
plt.show()

reference_calibration, reference_calibration_edges = build_binned_log_density_calibration(
    reference_log_p_truth, reference_log_p_flow
)
fig = plot_log_density_truth_binned(
    "balanced reference",
    reference_calibration,
    reference_calibration_edges,
    reference_log_p_truth,
    reference_log_p_flow,
)
plt.show()


## Train $p_S/p_{\rm ref}$ and $p_B/p_{\rm ref}$

We now return to the classifier-based likelihood-ratio trick from Exercises 2.2a and 2.2b. For each target sample, the numerator events have label one and flow-sampled reference events have label zero. The total BCE weight of each class is normalized to one, so

$$
q_\psi(y=1\mid x)=\frac{p_{\rm target}(x)}{p_{\rm target}(x)+p_{\rm ref}(x)},
\qquad
r_\psi(x)=\frac{q_\psi}{1-q_\psi}
\simeq\frac{p_{\rm target}(x)}{p_{\rm ref}(x)}.
$$

The signal and background networks are trained sequentially to keep peak memory bounded. Their calibration and reweighting validations use the same `density_ratio_trainer` methods as Exercise 2.


In [ ]:
def build_ratio_training_dataframe(target_df, reference_df, max_events, seed):
    n_events = min(int(max_events), len(target_df), len(reference_df))
    target = target_df.sample(n=n_events, random_state=seed)[FEATURES + ["weight"]].copy()
    reference = reference_df.sample(
        n=n_events, random_state=seed + 1
    )[FEATURES + ["weight"]].copy()

    target["weights"] = target["weight"]
    target["weights_normed"] = target["weight"] / target["weight"].sum()
    target["train_labels"] = 1.0
    reference["weights"] = reference["weight"]
    reference["weights_normed"] = reference["weight"] / reference["weight"].sum()
    reference["train_labels"] = 0.0

    return pd.concat([target, reference], ignore_index=True).sample(
        frac=1.0, random_state=seed + 2, ignore_index=True
    )


def as_inference_session(model_candidate):
    if isinstance(model_candidate, ort.InferenceSession):
        return model_candidate
    providers = [
        provider
        for provider in ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if provider in ort.get_available_providers()
    ]
    if not providers:
        providers = ort.get_available_providers()
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    return ort.InferenceSession(
        model_candidate.SerializeToString(),
        sess_options=options,
        providers=providers,
    )


def train_and_validate_ratio(sample_name, target_df, seed):
    training_dataframe = build_ratio_training_dataframe(
        target_df,
        reference_sample,
        MAX_RATIO_EVENTS_PER_CLASS,
        seed,
    )
    trainer = density_ratio_trainer(
        dataset=training_dataframe,
        weights=training_dataframe["weights_normed"],
        training_labels=training_dataframe["train_labels"],
        features=FEATURES,
        features_scaling=FEATURES,
        sample_name=[sample_name, "reference"],
        output_name="",
        path_to_figures=f"{RATIO_PLOT_DIR[sample_name]}/",
        path_to_models=f"{RATIO_MODEL_DIR[sample_name]}/",
    )

    trainer.train(
        hidden_layers=RATIO_HIDDEN_LAYERS,
        neurons=RATIO_NEURONS,
        number_of_epochs=RATIO_N_EPOCHS,
        batch_size=RATIO_BATCH_SIZE,
        learning_rate=RATIO_LEARNING_RATE,
        scalerType="MinMax",
        ensemble_index=0,
        verbose=1,
        rnd_seed=seed,
        holdout_split=RATIO_HOLDOUT_FRACTION,
        validation_split=RATIO_VALIDATION_FRACTION,
        callback_patience=RATIO_PATIENCE,
        num_workers=0,
        load_trained_models=RATIO_LOAD_IF_AVAILABLE,
        calibration=False,
        type_of_calibration="histogram",
        recalibrate_output=True,
        num_bins_cal=100,
    )

    # The same two validations used in Exercises 2.2a and 2.2b.
    trainer.make_calib_plots(observable="score", nbins=50, ensemble_index=0)
    trainer.make_reweighted_plots(FEATURES, "linear", 50)

    pack = {
        "scaler": trainer.scaler,
        "model": as_inference_session(trainer.model_NN),
    }
    del trainer, training_dataframe
    gc.collect()
    return pack


### Signal versus reference

The first classifier estimates $r_S(x)=p_S(x)/p_{\rm ref}(x)$. Its calibration plot tests the classifier-score interpretation, and its reweighting plots test whether $r_S(x)p_{\rm ref}(x)$ reproduces signal projections.


In [ ]:
ratio_models = {}
ratio_models["signal"] = train_and_validate_ratio(
    "signal", signal_train, SEED + 601
)
del signal_train
gc.collect()


### Background versus reference

The second classifier repeats the same construction for $r_B(x)=p_B(x)/p_{\rm ref}(x)$.


In [ ]:
ratio_models["background"] = train_and_validate_ratio(
    "background", background_train, SEED + 701
)
del background_train
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Normalize the ratios and construct weighted signal/background samples

For an exact density ratio,

$$
\mathbb E_{x\sim p_{\rm ref}}[r_S(x)]
=\mathbb E_{x\sim p_{\rm ref}}[r_B(x)]=1.
$$

The million-event reference sample lets us impose this normalization very precisely. We divide each predicted ratio by its reference-sample mean. The resulting per-event weights

$$w_i^S=\frac{r_S(x_i)}{N_{\rm ref}},\qquad
w_i^B=\frac{r_B(x_i)}{N_{\rm ref}}$$

sum to one and turn the same unweighted reference events into weighted signal or background samples.


In [ ]:
def evaluate_ratio(sample_name, dataframe, batch_size=100_000):
    pack = ratio_models[sample_name]
    chunks = []
    for start in range(0, len(dataframe), int(batch_size)):
        batch = dataframe.iloc[start : start + int(batch_size)][FEATURES]
        ratio = predict_with_model(
            batch.astype("float32", copy=False),
            scaler=pack["scaler"],
            model=pack["model"],
        )
        chunks.append(np.asarray(ratio, dtype=np.float64).reshape(-1))
    values = np.concatenate(chunks) if chunks else np.empty(0, dtype=np.float64)
    return np.maximum(values, RATIO_FLOOR)


RATIO_NORMALIZATION = {}
for sample_name in ["signal", "background"]:
    raw_ratio = evaluate_ratio(
        sample_name,
        reference_sample,
        batch_size=RATIO_EVALUATION_BATCH_SIZE,
    )
    RATIO_NORMALIZATION[sample_name] = float(raw_ratio.mean())
    normalized_ratio = raw_ratio / RATIO_NORMALIZATION[sample_name]
    reference_sample[f"ratio_{sample_name}"] = normalized_ratio
    reference_sample[f"weight_{sample_name}"] = normalized_ratio / len(reference_sample)
    component_weights = reference_sample[f"weight_{sample_name}"].to_numpy()
    effective_sample_size = 1.0 / np.sum(component_weights**2)
    ratio_quantiles = np.quantile(normalized_ratio, [0.001, 0.5, 0.999])

    print(
        f"{sample_name:10s}: raw E_ref[r] = {RATIO_NORMALIZATION[sample_name]:.6f}; "
        f"normalized weighted yield = "
        f"{reference_sample[f'weight_{sample_name}'].sum():.6f}; "
        f"ESS = {effective_sample_size:,.0f}/{len(reference_sample):,}; "
        f"ratio q(0.1%,50%,99.9%) = {ratio_quantiles}"
    )


In [ ]:
def plot_weighted_reference_closure(reference_df, signal_df, background_df, bins=50):
    fig, axes = plt.subplots(2, len(FEATURES), figsize=(3.2 * len(FEATURES), 6.0))
    comparisons = [
        ("signal", signal_df, "C4"),
        ("background", background_df, "C0"),
    ]
    for row, (sample_name, target_df, color) in enumerate(comparisons):
        for column, feature in enumerate(FEATURES):
            ax = axes[row, column]
            lo = min(target_df[feature].quantile(0.005), reference_df[feature].quantile(0.005))
            hi = max(target_df[feature].quantile(0.995), reference_df[feature].quantile(0.995))
            edges = np.linspace(lo, hi, bins + 1)
            ax.hist(
                reference_df[feature],
                bins=edges,
                weights=reference_df[f"weight_{sample_name}"],
                density=True,
                histtype="stepfilled",
                alpha=0.25,
                color=color,
                label="weighted reference",
            )
            ax.hist(
                target_df[feature],
                bins=edges,
                weights=target_df["weight"],
                density=True,
                histtype="step",
                lw=1.5,
                color="black",
                label=f"held-out {sample_name}",
            )
            ax.set_xlabel(feature)
            if column == 0:
                ax.set_ylabel("density")
            if row == 0 and column == 0:
                ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


fig = plot_weighted_reference_closure(
    reference_sample,
    signal_eval,
    background_eval,
)
plt.show()


## Reconstruct and validate the hybrid densities

For any selected event we now evaluate

$$
\log\hat p_S(x)=\log\hat p_{\rm ref}(x)+\log\hat r_S(x),
\qquad
\log\hat p_B(x)=\log\hat p_{\rm ref}(x)+\log\hat r_B(x).
$$

The same reference-flow error appears in both components. The ratio networks are trained against that very reference and can therefore correct its local distortions. The plots below compare the reconstructed densities with the analytic post-selection truth.


In [ ]:
asimov_dataset = pd.concat(
    [background_eval, signal_eval], ignore_index=True
).copy()
weights_asimov = asimov_dataset["weight"].to_numpy(dtype=np.float64)
X_asimov = asimov_dataset[FEATURES].to_numpy(dtype=np.float32)

lam_sig = float(TOTAL_YIELD["signal"])
lam_bkg = float(TOTAL_YIELD["background"])

log_p_ref_asimov = reference_log_prob_x(X_asimov)
HYBRID_RATIOS = {}
HYBRID_LOG_DENSITIES = {}
for sample_name in ["signal", "background"]:
    ratio = evaluate_ratio(
        sample_name,
        asimov_dataset,
        batch_size=RATIO_EVALUATION_BATCH_SIZE,
    ) / RATIO_NORMALIZATION[sample_name]
    HYBRID_RATIOS[sample_name] = ratio
    HYBRID_LOG_DENSITIES[sample_name] = log_p_ref_asimov + np.log(ratio)

TRUTH_LOG_DENSITIES = {
    "signal": selected_truth_log_density(X_asimov, "signal"),
    "background": selected_truth_log_density(X_asimov, "background"),
}

for sample_name in ["background", "signal"]:
    delta = HYBRID_LOG_DENSITIES[sample_name] - TRUTH_LOG_DENSITIES[sample_name]
    print(
        f"{sample_name:10s}: corr(log p) = "
        f"{np.corrcoef(TRUTH_LOG_DENSITIES[sample_name], HYBRID_LOG_DENSITIES[sample_name])[0, 1]:.4f}, "
        f"RMSE = {np.sqrt(np.mean(delta**2)):.4f}, bias = {delta.mean():.4f}"
    )
    fig = plot_log_density_truth_scatter(
        f"hybrid {sample_name}",
        TRUTH_LOG_DENSITIES[sample_name],
        HYBRID_LOG_DENSITIES[sample_name],
    )
    plt.show()

    calibration, edges = build_binned_log_density_calibration(
        TRUTH_LOG_DENSITIES[sample_name], HYBRID_LOG_DENSITIES[sample_name]
    )
    fig = plot_log_density_truth_binned(
        f"hybrid {sample_name}",
        calibration,
        edges,
        TRUTH_LOG_DENSITIES[sample_name],
        HYBRID_LOG_DENSITIES[sample_name],
    )
    plt.show()

np.save(HYBRID_DENSITY_DIR / "weights_asimov.npy", weights_asimov)
np.save(HYBRID_DENSITY_DIR / "log_p_ref_asimov.npy", log_p_ref_asimov)
np.save(HYBRID_DENSITY_DIR / "ratio_signal_asimov.npy", HYBRID_RATIOS["signal"])
np.save(HYBRID_DENSITY_DIR / "ratio_background_asimov.npy", HYBRID_RATIOS["background"])


## Extended likelihood from the hybrid densities

Using post-selection yields, the event intensity is

$$
\nu(x\mid\mu)=
\mu\lambda_S\hat p_{\rm ref}(x)\hat r_S(x)
+\lambda_B\hat p_{\rm ref}(x)\hat r_B(x).
$$

We deliberately evaluate the full hybrid densities in log space, exactly as in Exercise 4:

$$
-2\log L(\mu)=
-2\sum_i w_i\log\nu(x_i\mid\mu)
+2(\mu\lambda_S+\lambda_B).
$$

Factoring out $p_{\rm ref}(x_i)$ leaves a term independent of $\mu$. Consequently, the fit is controlled by the two high-precision ratios, while the explicit reference density remains available for density evaluation and event generation.


## Fit $\mu$ and construct the profile-likelihood-ratio test statistic

The normalizing flows provide the event densities, while JAX provides the likelihood gradient. Minimizing $\mathrm{NLL}=-2\log L$ gives the maximum-likelihood estimate $\hat\mu$. To test a fixed value of $\mu$, we use the **profile likelihood ratio**

$$
t_\mu = -2\log\lambda(\mu)
= -2\log\frac{L(\mu,\hat{\hat{\boldsymbol\theta}}_\mu)}{L(\hat\mu,\hat{\boldsymbol\theta})}
= \mathrm{NLL}(\mu,\hat{\hat{\boldsymbol\theta}}_\mu)-\mathrm{NLL}(\hat\mu,\hat{\boldsymbol\theta}).
$$

Here $\boldsymbol\theta$ denotes nuisance parameters, and $\hat{\hat{\boldsymbol\theta}}_\mu$ is their conditional best fit at fixed $\mu$. This exercise has no nuisance parameters, so the profiling is trivial. The scan below constructs the test-statistic curve $t_\mu$, whose minimum is $t_{\hat\mu}=0$.


In [ ]:
EPS = 1.0e-300
INVALID_NLL = 1.0e30


def make_direct_log_density_nll(
    log_p_sig_np,
    log_p_bkg_np,
    weights_np,
    lam_sig_value,
    lam_bkg_value,
):
    log_sig_intensity = jnp.asarray(
        np.log(float(lam_sig_value)) + np.asarray(log_p_sig_np, dtype=np.float64)
    )
    log_bkg_intensity = jnp.asarray(
        np.log(float(lam_bkg_value)) + np.asarray(log_p_bkg_np, dtype=np.float64)
    )
    weights_j = jnp.asarray(np.asarray(weights_np, dtype=np.float64))

    def _nll(params):
        mu = params[0]
        safe_mu = jnp.maximum(mu, EPS)
        log_event_intensity = jnp.logaddexp(
            log_bkg_intensity,
            jnp.log(safe_mu) + log_sig_intensity,
        )
        n_expected = mu * float(lam_sig_value) + float(lam_bkg_value)
        nll = -2.0 * jnp.sum(weights_j * log_event_intensity) + 2.0 * n_expected
        invalid = (mu < 0.0) | (n_expected <= 0.0) | (~jnp.isfinite(nll))
        negative_mu = jnp.minimum(mu, 0.0)
        return jnp.where(
            invalid,
            INVALID_NLL * (1.0 + negative_mu**2),
            nll,
        )

    return _nll


nll_hybrid = make_direct_log_density_nll(
    HYBRID_LOG_DENSITIES["signal"],
    HYBRID_LOG_DENSITIES["background"],
    weights_asimov,
    lam_sig,
    lam_bkg,
)
inf_hybrid = nsbi_common_utils.inference.inference(
    model_nll=jax.jit(nll_hybrid),
    initial_values=[1.0],
    list_parameters=["mu"],
    num_unconstrained_params=1,
    model_grad=jax.jit(jax.grad(nll_hybrid)),
)

print("\n" + "=" * 40)
print(" HYBRID FLOW + RATIO FIT RESULTS ")
print("=" * 40 + "\n")
inf_hybrid.perform_fit(freeze_params=[])
mu_min_hybrid = inf_hybrid.pulls_global_fit

SCAN_RANGE = (0.0, 20.0)
scan_hybrid, tmu_hybrid = inf_hybrid.perform_profile_scan(
    parameter_name="mu",
    freeze_params=[],
    bound_range=SCAN_RANGE,
    fit_strategy=0,
    size=50,
)
fig = plot_profile_scan(scan_hybrid, tmu_hybrid, label="Hybrid densities")
plt.show()


### Compare with the analytic post-selection likelihood

The analytic fit uses the same weighted evaluation sample and post-selection yields. Any difference between the curves is therefore caused by the learned reference/ratio construction rather than by finite evaluation-sample fluctuations.


In [ ]:
nll_truth = make_direct_log_density_nll(
    TRUTH_LOG_DENSITIES["signal"],
    TRUTH_LOG_DENSITIES["background"],
    weights_asimov,
    lam_sig,
    lam_bkg,
)
inf_truth = nsbi_common_utils.inference.inference(
    model_nll=jax.jit(nll_truth),
    initial_values=[1.0],
    list_parameters=["mu"],
    num_unconstrained_params=1,
    model_grad=jax.jit(jax.grad(nll_truth)),
)

print("\n" + "=" * 40)
print(" ANALYTIC RECO-DENSITY FIT RESULTS ")
print("=" * 40 + "\n")
inf_truth.perform_fit(freeze_params=[])
mu_min_truth = inf_truth.pulls_global_fit
scan_truth, tmu_truth = inf_truth.perform_profile_scan(
    parameter_name="mu",
    freeze_params=[],
    bound_range=SCAN_RANGE,
    fit_strategy=0,
    size=50,
)

mu_hat_hybrid = float(np.asarray(mu_min_hybrid).reshape(-1)[0])
mu_hat_truth = float(np.asarray(mu_min_truth).reshape(-1)[0])
print(
    f"Fitted minima: hybrid mu_hat = {mu_hat_hybrid:.6f}; "
    f"analytic mu_hat = {mu_hat_truth:.6f}; "
    f"hybrid - truth = {mu_hat_hybrid - mu_hat_truth:+.6f}"
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(scan_hybrid, tmu_hybrid, lw=2, label="Hybrid flow + ratios")
ax.plot(
    scan_truth,
    tmu_truth,
    lw=2,
    ls="--",
    label="Analytic reco densities",
)
ax.axvline(1.0, color="black", ls=":", lw=1, alpha=0.7)
ax.set_ylim(bottom=0)
ax.set_xlabel(r"$\mu_{\rm signal}$")
ax.set_ylabel(r"$t_\mu$")
ax.legend()
fig.tight_layout()
plt.show()


## Sampling distributions with weighted reference sampling

The fixed reference sample represents $p_{\rm ref}$. Its normalized columns `weight_signal` and `weight_background` represent $p_S$ and $p_B$. For each toy we:

1. draw independent Poisson signal and background counts;
2. sample reference-event indices with probabilities proportional to the appropriate ratio weights;
3. fit the resulting unweighted events with the same hybrid likelihood.

Only the cached ratio

$$
q_i=\frac{\lambda_S r_S(x_i)}{\lambda_B r_B(x_i)}
$$

is required by the likelihood-ratio fit. Thus the flow and neural networks are evaluated once, not once per toy.


In [ ]:
N_TOYS = 100
TOY_HYPOTHESES = (0.0, 1.0)
TOY_SEED = 314159
TOY_PLOT_BINS = 35
TOY_VERBOSE_FITS = False
TOY_PADDING_SIGMAS = 8.0

TOY_MAX_EXPECTED_EVENTS = max(
    mu * lam_sig + lam_bkg for mu in TOY_HYPOTHESES
)
TOY_PAD_SIZE = int(
    np.ceil(
        TOY_MAX_EXPECTED_EVENTS
        + TOY_PADDING_SIGMAS * np.sqrt(TOY_MAX_EXPECTED_EVENTS)
        + 32
    )
)
print(f"JAX toy padding size: {TOY_PAD_SIZE:,} events")


def normalized_cdf(weights):
    weights = np.asarray(weights, dtype=np.float64)
    cdf = np.cumsum(weights / weights.sum())
    cdf[-1] = 1.0
    return cdf


SIGNAL_REFERENCE_CDF = normalized_cdf(reference_sample["weight_signal"])
BACKGROUND_REFERENCE_CDF = normalized_cdf(reference_sample["weight_background"])
REFERENCE_LOG_Q = (
    np.log(lam_sig / lam_bkg)
    + np.log(reference_sample["ratio_signal"].to_numpy(dtype=np.float64))
    - np.log(reference_sample["ratio_background"].to_numpy(dtype=np.float64))
)
REFERENCE_Q = np.exp(np.clip(REFERENCE_LOG_Q, -80.0, 80.0))


def draw_weighted_indices(cdf, n_events, rng):
    if int(n_events) == 0:
        return np.empty(0, dtype=np.int64)
    return np.searchsorted(cdf, rng.random(int(n_events)), side="right")


def generate_hybrid_toy_q(mu_true, lam_sig_value, lam_bkg_value, rng):
    mu_true = float(mu_true)
    if mu_true < 0.0:
        raise ValueError("mu_true must be non-negative.")

    n_signal = int(rng.poisson(mu_true * lam_sig_value))
    n_background = int(rng.poisson(lam_bkg_value))
    signal_indices = draw_weighted_indices(SIGNAL_REFERENCE_CDF, n_signal, rng)
    background_indices = draw_weighted_indices(
        BACKGROUND_REFERENCE_CDF, n_background, rng
    )
    q = np.concatenate(
        [REFERENCE_Q[background_indices], REFERENCE_Q[signal_indices]]
    )
    return q, n_signal + n_background, n_signal, n_background


def _toy_relative_nll(params, q_padded, lam_sig_value, lam_bkg_value):
    mu = params[0]
    event_factor = 1.0 + mu * q_padded
    n_expected = mu * lam_sig_value + lam_bkg_value
    nll = 2.0 * (
        mu * lam_sig_value
        - jnp.sum(jnp.log(jnp.maximum(event_factor, EPS)))
    )
    invalid = (n_expected <= 0.0) | jnp.any(event_factor <= 0.0)
    return jnp.where(invalid, 1.0e30 + 1.0e12 * mu**2, nll)


TOY_NLL_KERNEL = jax.jit(_toy_relative_nll)
TOY_GRAD_KERNEL = jax.jit(jax.grad(_toy_relative_nll, argnums=0))


def fit_hybrid_toy(q, test_mu, lam_sig_value, lam_bkg_value):
    if len(q) == 0:
        return 0.0, 2.0 * float(test_mu) * lam_sig_value, 0.0
    if len(q) > TOY_PAD_SIZE:
        raise RuntimeError(
            f"Toy has {len(q):,} events, above TOY_PAD_SIZE={TOY_PAD_SIZE:,}."
        )

    q_padded = np.zeros(TOY_PAD_SIZE, dtype=np.float64)
    q_padded[: len(q)] = q
    q_jax = jnp.asarray(q_padded)

    def toy_nll(params):
        return TOY_NLL_KERNEL(params, q_jax, lam_sig_value, lam_bkg_value)

    def toy_grad(params):
        return TOY_GRAD_KERNEL(params, q_jax, lam_sig_value, lam_bkg_value)

    toy_inference = nsbi_common_utils.inference.inference(
        model_nll=toy_nll,
        initial_values=[max(0.1, float(test_mu))],
        list_parameters=["mu"],
        num_unconstrained_params=1,
        model_grad=toy_grad,
    )
    fit_output = nullcontext() if TOY_VERBOSE_FITS else redirect_stdout(StringIO())
    with fit_output:
        toy_inference.perform_fit(fit_strategy=0, freeze_params=[])
    mu_unconstrained = float(
        np.asarray(toy_inference.pulls_global_fit).reshape(-1)[0]
    )
    mu_hat = max(0.0, mu_unconstrained)

    test_mu = float(test_mu)
    t_mu = 2.0 * (
        (test_mu - mu_hat) * lam_sig_value
        - np.sum(np.log1p(test_mu * q) - np.log1p(mu_hat * q))
    )
    t_mu = max(0.0, float(t_mu))
    response = q / (1.0 + test_mu * q)
    observed_information = float(np.sum(response**2))
    return mu_hat, t_mu, observed_information


def run_hybrid_toys(mu_true, n_toys, lam_sig_value, lam_bkg_value, seed):
    rng = np.random.default_rng(seed)
    rows = []
    progress_every = max(1, int(n_toys) // 10)
    for toy_index in range(int(n_toys)):
        q, n_total, n_signal, n_background = generate_hybrid_toy_q(
            mu_true, lam_sig_value, lam_bkg_value, rng
        )
        mu_hat, t_mu, information = fit_hybrid_toy(
            q, mu_true, lam_sig_value, lam_bkg_value
        )
        rows.append(
            {
                "mu_true": float(mu_true),
                "toy": toy_index,
                "n_events": n_total,
                "n_signal": n_signal,
                "n_background": n_background,
                "mu_hat": mu_hat,
                "t_mu": t_mu,
                "information": information,
            }
        )
        if (toy_index + 1) % progress_every == 0:
            print(f"mu={mu_true:g}: completed {toy_index + 1}/{n_toys} toys")
    return pd.DataFrame(rows)


def asymptotic_sigmas(toy_results):
    return {
        float(mu_true): 1.0 / np.sqrt(float(group["information"].mean()))
        for mu_true, group in toy_results.groupby("mu_true")
    }


toy_results = pd.concat(
    [
        run_hybrid_toys(
            mu_true,
            N_TOYS,
            lam_sig,
            lam_bkg,
            seed=TOY_SEED + int(1000 * mu_true),
        )
        for mu_true in TOY_HYPOTHESES
    ],
    ignore_index=True,
)

sigma_by_mu = asymptotic_sigmas(toy_results)
print("Asymptotic sigma_mu estimates:", sigma_by_mu)
print(
    toy_results.groupby("mu_true")[["mu_hat", "t_mu", "n_events"]]
    .agg(["mean", "std"])
    .to_string()
)

fig = plot_t_mu_toys(toy_results, n_bins=TOY_PLOT_BINS)
plt.show()
fig = plot_mu_hat_toys(toy_results, sigma_by_mu, n_bins=TOY_PLOT_BINS)
plt.show()


## Suggested exercises

1. Compare the hybrid fit with Exercise 4 using the same PRESEL cut and evaluation sample.
2. Toggle `USE_QUADRATIC_SPLINE` and check whether residual reference-flow differences affect the hybrid densities or cancel through the ratios.
3. Change the reference mixture away from 50:50 and study density-ratio calibration in signal-like and background-like regions.
4. Remove the explicit ratio-normalization step and measure its effect on the extended-likelihood minimum.
5. Increase the density-ratio ensemble size and compare the spread of the fitted $\hat\mu$ across independently trained networks.
